In [1]:
import requests
from lakehouse.daft import bronze, silver
import json
import daft

In [2]:
CATALOG = "daft_catalog"

# 1. Set Up and Bronze Data

In [3]:
options = {"catalog": CATALOG, "target_schema": "bronze"}

In [4]:
@daft.udf(return_dtype=daft.DataType.string())
def get_properties(urls: daft.Series) -> list:
    result = []
    for url in urls.to_pylist():
        json_request = requests.get(url).json()
        result.append(json.dumps(json_request["result"]["properties"]))
    return result

In [5]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return daft.from_pylist(results)

    def custom_transform(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        return df.with_column("properties", get_properties(daft.col("url")))

    def target_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.target_schema}/{table}"


bronze_instance = StarWarsBronze(**options)

In [6]:
bronze_instance.load().transform().write(mode="overwrite").execute("people", "planets")

2025-03-30 15:04:16 | people | execute | Started
2025-03-30 15:04:16 | people | load | Started
2025-03-30 15:04:21 | people | load | Completed in 0.07 min
2025-03-30 15:04:21 | people | transform | Started
2025-03-30 15:04:21 | people | transform | Completed in 0.0 min
2025-03-30 15:04:21 | people | write | Started
c:\Users\nikol\miniconda3\envs\lh-exec\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


                                                           d

2025-03-30 15:04:55 | people | write | Completed in 0.55 min
2025-03-30 15:04:55 | people | execute | Completed in 0.63 min
2025-03-30 15:04:55 | planets | execute | Started
2025-03-30 15:04:55 | planets | load | Started


2025-03-30 15:04:58 | planets | load | Completed in 0.03 min
2025-03-30 15:04:58 | planets | transform | Started
2025-03-30 15:04:58 | planets | transform | Completed in 0.0 min
2025-03-30 15:04:58 | planets | write | Started


                                                           d

2025-03-30 15:05:20 | planets | write | Completed in 0.37 min
2025-03-30 15:05:20 | planets | execute | Completed in 0.42 min


In [7]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/bronze/people")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidUtf8,urlUtf8,propertiesUtf8
2025-03-30 15:04:21.488309,Luke Skywalker,1,https://www.swapi.tech/api/people/1,"{""created"": ""2025-03-30T08:18:34.423Z"", ""edited"": ""2025-03-30T08:18:34.423Z"", ""name"": ""Luke Skywalker"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""blond"", ""height"": ""172"", ""eye_color"": ""blue"", ""mass"": ""77"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/1""}"
2025-03-30 15:04:21.488309,C-3PO,2,https://www.swapi.tech/api/people/2,"{""created"": ""2025-03-30T08:18:34.423Z"", ""edited"": ""2025-03-30T08:18:34.423Z"", ""name"": ""C-3PO"", ""gender"": ""n/a"", ""skin_color"": ""gold"", ""hair_color"": ""n/a"", ""height"": ""167"", ""eye_color"": ""yellow"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""112BBY"", ""url"": ""https://www.swapi.tech/api/people/2""}"
2025-03-30 15:04:21.488309,R2-D2,3,https://www.swapi.tech/api/people/3,"{""created"": ""2025-03-30T08:18:34.423Z"", ""edited"": ""2025-03-30T08:18:34.423Z"", ""name"": ""R2-D2"", ""gender"": ""n/a"", ""skin_color"": ""white, blue"", ""hair_color"": ""n/a"", ""height"": ""96"", ""eye_color"": ""red"", ""mass"": ""32"", ""homeworld"": ""https://www.swapi.tech/api/planets/8"", ""birth_year"": ""33BBY"", ""url"": ""https://www.swapi.tech/api/people/3""}"
2025-03-30 15:04:21.488309,Darth Vader,4,https://www.swapi.tech/api/people/4,"{""created"": ""2025-03-30T08:18:34.423Z"", ""edited"": ""2025-03-30T08:18:34.423Z"", ""name"": ""Darth Vader"", ""gender"": ""male"", ""skin_color"": ""white"", ""hair_color"": ""none"", ""height"": ""202"", ""eye_color"": ""yellow"", ""mass"": ""136"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""41.9BBY"", ""url"": ""https://www.swapi.tech/api/people/4""}"
2025-03-30 15:04:21.488309,Leia Organa,5,https://www.swapi.tech/api/people/5,"{""created"": ""2025-03-30T08:18:34.423Z"", ""edited"": ""2025-03-30T08:18:34.423Z"", ""name"": ""Leia Organa"", ""gender"": ""female"", ""skin_color"": ""light"", ""hair_color"": ""brown"", ""height"": ""150"", ""eye_color"": ""brown"", ""mass"": ""49"", ""homeworld"": ""https://www.swapi.tech/api/planets/2"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/5""}"
2025-03-30 15:04:21.488309,Owen Lars,6,https://www.swapi.tech/api/people/6,"{""created"": ""2025-03-30T08:18:34.423Z"", ""edited"": ""2025-03-30T08:18:34.423Z"", ""name"": ""Owen Lars"", ""gender"": ""male"", ""skin_color"": ""light"", ""hair_color"": ""brown, grey"", ""height"": ""178"", ""eye_color"": ""blue"", ""mass"": ""120"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""52BBY"", ""url"": ""https://www.swapi.tech/api/people/6""}"
2025-03-30 15:04:21.488309,Beru Whitesun lars,7,https://www.swapi.tech/api/people/7,"{""created"": ""2025-03-30T08:18:34.423Z"", ""edited"": ""2025-03-30T08:18:34.423Z"", ""name"": ""Beru Whitesun lars"", ""gender"": ""female"", ""skin_color"": ""light"", ""hair_color"": ""brown"", ""height"": ""165"", ""eye_color"": ""blue"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""47BBY"", ""url"": ""https://www.swapi.tech/api/people/7""}"
2025-03-30 15:04:21.488309,R5-D4,8,https://www.swapi.tech/api/people/8,"{""created"": ""2025-03-30T08:18:34.423Z"", ""edited"": ""2025-03-30T08:18:34.423Z"", ""name"": ""R5-D4"", ""gender"": ""n/a"", ""skin_color"": ""white, red"", ""hair_color"": ""n/a"", ""height"": ""97"", ""eye_color"": ""red"", ""mass"": ""32"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""unknown"", ""url"": ""https://www.swapi.tech/api/people/8""}"


No. Rows: 82


# 2 Silver

In [8]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

In [9]:
class StarWarsSilver(silver.Silver):
    def custom_filter(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        return df.where("uid <= '25'")

    def custom_transform(self, df: daft.DataFrame, table: str) -> daft.DataFrame:
        df = df.with_column("uid", daft.col("uid").cast("int"))
        return df

    def source_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.source_schema}/{table}"

    def target_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.target_schema}/{table}"


silver_instance = StarWarsSilver(
    catalog=CATALOG, source_schema="bronze", target_schema="silver"
)

In [10]:
silver_instance.load(filter="custom").transform().write(mode="overwrite").execute(
    "people", "planets"
)

2025-03-30 15:05:30 | people | execute | Started
2025-03-30 15:05:30 | people | load | Started
2025-03-30 15:05:30 | people | load | Completed in 0.0 min
2025-03-30 15:05:30 | people | transform | Started
2025-03-30 15:05:30 | people | transform | Completed in 0.0 min
2025-03-30 15:05:30 | people | write | Started
2025-03-30 15:05:30 | people | write | Completed in 0.0 min
2025-03-30 15:05:30 | people | execute | Completed in 0.0 min
2025-03-30 15:05:30 | planets | execute | Started
2025-03-30 15:05:30 | planets | load | Started
2025-03-30 15:05:30 | planets | load | Completed in 0.0 min
2025-03-30 15:05:30 | planets | transform | Started
2025-03-30 15:05:30 | planets | transform | Completed in 0.0 min
2025-03-30 15:05:30 | planets | write | Started
2025-03-30 15:05:30 | planets | write | Completed in 0.0 min
2025-03-30 15:05:30 | planets | execute | Completed in 0.0 min


In [11]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/silver/people")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_SilverTSTimestamp(Microseconds, None)","LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidInt32,urlUtf8,propertiesUtf8
2025-03-30 15:05:30.569264,2025-03-30 15:04:21.488309,Luke Skywalker,1,https://www.swapi.tech/api/people/1,"{""created"": ""2025-03-30T08:18:34.423Z"", ""edited"": ""2025-03-30T08:18:34.423Z"", ""name"": ""Luke Skywalker"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""blond"", ""height"": ""172"", ""eye_color"": ""blue"", ""mass"": ""77"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""19BBY"", ""url"": ""https://www.swapi.tech/api/people/1""}"
2025-03-30 15:05:30.569264,2025-03-30 15:04:21.488309,C-3PO,2,https://www.swapi.tech/api/people/2,"{""created"": ""2025-03-30T08:18:34.423Z"", ""edited"": ""2025-03-30T08:18:34.423Z"", ""name"": ""C-3PO"", ""gender"": ""n/a"", ""skin_color"": ""gold"", ""hair_color"": ""n/a"", ""height"": ""167"", ""eye_color"": ""yellow"", ""mass"": ""75"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""112BBY"", ""url"": ""https://www.swapi.tech/api/people/2""}"
2025-03-30 15:05:30.569264,2025-03-30 15:04:21.488309,Obi-Wan Kenobi,10,https://www.swapi.tech/api/people/10,"{""created"": ""2025-03-30T08:18:34.423Z"", ""edited"": ""2025-03-30T08:18:34.423Z"", ""name"": ""Obi-Wan Kenobi"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""auburn, white"", ""height"": ""182"", ""eye_color"": ""blue-gray"", ""mass"": ""77"", ""homeworld"": ""https://www.swapi.tech/api/planets/20"", ""birth_year"": ""57BBY"", ""url"": ""https://www.swapi.tech/api/people/10""}"
2025-03-30 15:05:30.569264,2025-03-30 15:04:21.488309,Anakin Skywalker,11,https://www.swapi.tech/api/people/11,"{""created"": ""2025-03-30T08:18:34.423Z"", ""edited"": ""2025-03-30T08:18:34.423Z"", ""name"": ""Anakin Skywalker"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""blond"", ""height"": ""188"", ""eye_color"": ""blue"", ""mass"": ""84"", ""homeworld"": ""https://www.swapi.tech/api/planets/1"", ""birth_year"": ""41.9BBY"", ""url"": ""https://www.swapi.tech/api/people/11""}"
2025-03-30 15:05:30.569264,2025-03-30 15:04:21.488309,Wilhuff Tarkin,12,https://www.swapi.tech/api/people/12,"{""created"": ""2025-03-30T08:18:34.423Z"", ""edited"": ""2025-03-30T08:18:34.423Z"", ""name"": ""Wilhuff Tarkin"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""auburn, grey"", ""height"": ""180"", ""eye_color"": ""blue"", ""mass"": ""unknown"", ""homeworld"": ""https://www.swapi.tech/api/planets/21"", ""birth_year"": ""64BBY"", ""url"": ""https://www.swapi.tech/api/people/12""}"
2025-03-30 15:05:30.569264,2025-03-30 15:04:21.488309,Chewbacca,13,https://www.swapi.tech/api/people/13,"{""created"": ""2025-03-30T08:18:34.423Z"", ""edited"": ""2025-03-30T08:18:34.423Z"", ""name"": ""Chewbacca"", ""gender"": ""male"", ""skin_color"": ""unknown"", ""hair_color"": ""brown"", ""height"": ""228"", ""eye_color"": ""blue"", ""mass"": ""112"", ""homeworld"": ""https://www.swapi.tech/api/planets/14"", ""birth_year"": ""200BBY"", ""url"": ""https://www.swapi.tech/api/people/13""}"
2025-03-30 15:05:30.569264,2025-03-30 15:04:21.488309,Han Solo,14,https://www.swapi.tech/api/people/14,"{""created"": ""2025-03-30T08:18:34.423Z"", ""edited"": ""2025-03-30T08:18:34.423Z"", ""name"": ""Han Solo"", ""gender"": ""male"", ""skin_color"": ""fair"", ""hair_color"": ""brown"", ""height"": ""180"", ""eye_color"": ""brown"", ""mass"": ""80"", ""homeworld"": ""https://www.swapi.tech/api/planets/22"", ""birth_year"": ""29BBY"", ""url"": ""https://www.swapi.tech/api/people/14""}"
2025-03-30 15:05:30.569264,2025-03-30 15:04:21.488309,Greedo,15,https://www.swapi.tech/api/people/15,"{""created"": ""2025-03-30T08:18:34.423Z"", ""edited"": ""2025-03-30T08:18:34.423Z"", ""name"": ""Greedo"", ""gender"": ""male"", ""skin_color"": ""green"", ""hair_color"": ""n/a"", ""height"": ""173"", ""eye_color"": ""black"", ""mass"": ""74"", ""homeworld"": ""https

No. Rows: 17


# 3 Optimize Silver

In [ ]:
# Run optimize and vacuum with default
silver_instance.optimize(optimize=True, vacuum=True).execute("people")

2025-03-16 07:37:06 | people | execute | Started
2025-03-16 07:37:06 | people | execute | Started
2025-03-16 07:37:06 | people | load | Started
2025-03-16 07:37:06 | people | load | Completed in 0.0 min
2025-03-16 07:37:06 | people | transform | Started
2025-03-16 07:37:06 | people | transform | Completed in 0.0 min
2025-03-16 07:37:06 | people | write | Started
2025-03-16 07:37:08 | people | write | Completed in 0.02 min
2025-03-16 07:37:08 | people | tblproperties | Started
2025-03-16 07:37:23 | people | tblproperties | Completed in 0.25 min
2025-03-16 07:37:23 | people | optimize | Started
2025-03-16 07:37:46 | people | optimize | Completed in 0.37 min
2025-03-16 07:37:46 | people | execute | Completed in 0.65 min
2025-03-16 07:37:46 | people | execute | Completed in 0.65 min


In [ ]:
# You can also do this with the default class
(silver.Silver(**options).optimize(optimize=True, vacuum=True).execute("people"))

2025-03-16 07:37:46 | people | execute | Started
2025-03-16 07:37:46 | people | execute | Started
2025-03-16 07:37:46 | people | tblproperties | Started
2025-03-16 07:37:46 | people | tblproperties | Completed in 0.0 min
2025-03-16 07:37:46 | people | optimize | Started
2025-03-16 07:38:04 | people | optimize | Completed in 0.28 min
2025-03-16 07:38:04 | people | execute | Completed in 0.3 min
2025-03-16 07:38:04 | people | execute | Completed in 0.3 min


In [ ]:
# run above commands also together with the write
(
    silver_instance.load(filter="custom")
    .transform()
    .write(mode="overwrite")
    .optimize(optimize=True, vacuum=True)
    .execute("people")
)

2025-03-16 07:38:04 | people | execute | Started
2025-03-16 07:38:04 | people | execute | Started
2025-03-16 07:38:04 | people | load | Started
2025-03-16 07:38:04 | people | load | Completed in 0.0 min
2025-03-16 07:38:04 | people | transform | Started
2025-03-16 07:38:04 | people | transform | Completed in 0.0 min
2025-03-16 07:38:04 | people | write | Started
2025-03-16 07:38:07 | people | write | Completed in 0.03 min
2025-03-16 07:38:07 | people | tblproperties | Started
2025-03-16 07:38:09 | people | tblproperties | Completed in 0.02 min
2025-03-16 07:38:09 | people | optimize | Started
2025-03-16 07:38:29 | people | optimize | Completed in 0.32 min
2025-03-16 07:38:29 | people | execute | Completed in 0.4 min
2025-03-16 07:38:29 | people | execute | Completed in 0.4 min


In [ ]:
# run Optimize with ZOrder (liquid not supported yet by deltalake) and define a retention time for vacuum in hours

silver_instance.optimize(
    optimize=True, z_order_cols=["gender"], vacuum=True, retention=168
).execute("people")

2025-03-16 07:38:29 | people | execute | Started
2025-03-16 07:38:29 | people | execute | Started
2025-03-16 07:38:29 | people | load | Started
2025-03-16 07:38:29 | people | load | Completed in 0.0 min
2025-03-16 07:38:29 | people | transform | Started
2025-03-16 07:38:29 | people | transform | Completed in 0.0 min
2025-03-16 07:38:29 | people | write | Started
2025-03-16 07:38:32 | people | write | Completed in 0.05 min
2025-03-16 07:38:32 | people | tblproperties | Started
2025-03-16 07:38:34 | people | tblproperties | Completed in 0.02 min
2025-03-16 07:38:34 | people | optimize | Started
2025-03-16 07:38:44 | people | optimize | Completed in 0.15 min
2025-03-16 07:38:44 | people | execute | Completed in 0.23 min
2025-03-16 07:38:44 | people | execute | Completed in 0.23 min


# Review in Delta History and details

In [25]:
from deltalake import DeltaTable

path = f"D:/Data/{CATALOG}/{options['target_schema']}/people"
dt = DeltaTable(path)
dt.history()

[{'timestamp': 1743339930616,
  'operation': 'CREATE TABLE',
  'operationParameters': {'mode': 'ErrorIfExists',
   'protocol': '{"minReaderVersion":3,"minWriterVersion":7,"readerFeatures":["timestampNtz"],"writerFeatures":["timestampNtz"]}',
   'location': 'file:///D:/Data/daft_catalog/silver/people',
   'metadata': '{"configuration":{},"createdTime":1743339930616,"description":null,"format":{"options":{},"provider":"parquet"},"id":"9aa804a8-8b31-447e-b2e7-050bc4d5e466","name":null,"partitionColumns":[],"schemaString":"{\\"type\\":\\"struct\\",\\"fields\\":[{\\"name\\":\\"LH_SilverTS\\",\\"type\\":\\"timestamp_ntz\\",\\"nullable\\":true,\\"metadata\\":{}},{\\"name\\":\\"LH_BronzeTS\\",\\"type\\":\\"timestamp_ntz\\",\\"nullable\\":true,\\"metadata\\":{}},{\\"name\\":\\"name\\",\\"type\\":\\"string\\",\\"nullable\\":true,\\"metadata\\":{}},{\\"name\\":\\"uid\\",\\"type\\":\\"integer\\",\\"nullable\\":true,\\"metadata\\":{}},{\\"name\\":\\"url\\",\\"type\\":\\"string\\",\\"nullable\\":tru

# 4 Clean Up

In [ ]:
import shutil

shutil.rmtree(f"D:/Data/{CATALOG}")

DataFrame[]